# New UniversalXAS heads from exported E2E features + VASP encoder adaptation

This notebook starts from `fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42`.

Flow:

1. Load and validate the exported 64D encoder features.
2. Train two new `UniversalXAS` FEFF heads with different dropout/scheduler choices.
3. Select the UniversalXAS head by validation eta, then fine-tune it for every FEFF element.
4. **Later sections:** fine-tune the encoder on VASP with the provided UniversalXAS head **not frozen**.
5. Freeze/export that VASP-tuned encoder, then fine-tune a UniversalXAS head on the new VASP feature space.

Validation metrics drive selection. Test evaluation is gated by `RUN_FINAL_TEST` so it is not accidentally used during model selection.


## 1. Imports and configuration

Set `OMNIXAS_REPO_ROOT`, `OMNIXAS_E2E_UNIVERSAL_RUN`, or `OMNIXAS_DATA_ROOT` if your paths differ.


In [1]:
from __future__ import annotations

from pathlib import Path
import json
import os
import random
import shutil
import sys

import numpy as np
import pandas as pd
import torch
import lightning.pytorch as pl
from IPython.display import display
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from torch.utils.data import DataLoader

def find_repo_root() -> Path:
    env = os.environ.get("OMNIXAS_REPO_ROOT")
    candidates = []
    if env:
        candidates.append(Path(env).expanduser())
    candidates += [Path.cwd(), *Path.cwd().parents]
    candidates += [
        Path.home() / "Desktop" / "OmniXAS",
        Path("/mnt/c/Users/anton/Desktop/OmniXAS"),
        Path("/c/Users/anton/Desktop/OmniXAS"),
    ]
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / "pyproject.toml").exists() and (candidate / "omnixas").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find OmniXAS repo root. Start Jupyter from the repo, or set OMNIXAS_REPO_ROOT. "
        f"cwd={Path.cwd()}"
    )

REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "tutorial_omnixas"))

from omnixas.data.ml_data import MLData, MLSplits
from omnixas.model.xasblock import XASBlock
from omnixas.model.xasblock_regressor import XASBlockRegressor

from train_all8_feff import (
    BATCH,
    ENCODER_BATCH,
    ENCODER_DERIV_LAMBDA,
    EXPORT_TASKS,
    FEFF_TASKS,
    VASP_TASKS,
    FEATURE_SCALE,
    HEAD_WIDTHS,
    INPUT_DIM,
    OUTPUT_DIM,
    SPLITS,
    CollateGraphs,
    FEFFDataset,
    checkpoint_score,
    eval_ckpt,
    patch_matgl_gpu_constants,
    torch_load,
    validate_raw_structures,
)
from train_e2e_custom_universal import EndToEndUniversal, LitEndToEnd

pl.seed_everything(43, workers=True)
np.random.seed(43); random.seed(43); torch.manual_seed(43)
torch.set_float32_matmul_precision("medium")

ARCHIVE_RUN = Path(
    os.environ.get(
        "OMNIXAS_E2E_UNIVERSAL_RUN",
        REPO_ROOT.parent / "fulltrainingcopy072726" / "m3gnetAll8E2EUniversal" / "e2e_universal_seed42",
    )
).expanduser().resolve()
FEATURES_DIR = ARCHIVE_RUN / "features"
NOTEBOOK_RUN_ROOT = ARCHIVE_RUN / "notebook_new_universal_runs"
NOTEBOOK_RUN_ROOT.mkdir(parents=True, exist_ok=True)

RUN_TRAIN_UNIVERSAL = True
RUN_TUNE_FEFF = True
RUN_FINAL_TEST = True  # run final test after validation selection in this batch job
RESUME_EXISTING = True

UNIVERSAL_VARIANTS = [
    {"name": "feff_cosT800_lr7e-4_do025_seed43", "seed": 43, "dropout": 0.25, "lr": 7e-4, "scheduler": "cosine", "epochs": 800, "patience": 60, "batch_size": 32, "cosine_t": 800},
    {"name": "feff_plateau_lr5e-4_do010_seed44", "seed": 44, "dropout": 0.10, "lr": 5e-4, "scheduler": "plateau", "epochs": 800, "patience": 60, "batch_size": 32, "plateau_patience": 8},
]
FEFF_TUNING_SWEEP = [
    {"name": "cosT250_lr1e-4_do0p0", "seed": 143, "dropout": 0.00, "lr": 1e-4, "scheduler": "cosine", "epochs": 1000, "patience": 25, "cosine_t": 250},
    {"name": "cosT250_lr1e-4_do0p1", "seed": 144, "dropout": 0.10, "lr": 1e-4, "scheduler": "cosine", "epochs": 1000, "patience": 25, "cosine_t": 250},
    {"name": "cosT500_lr3e-4_do0p1", "seed": 145, "dropout": 0.10, "lr": 3e-4, "scheduler": "cosine", "epochs": 1000, "patience": 25, "cosine_t": 500},
]

print("repo:", REPO_ROOT)
print("archive run:", ARCHIVE_RUN)
print("features:", FEATURES_DIR)
print("new output root:", NOTEBOOK_RUN_ROOT)


Seed set to 43


repo: /mnt/c/Users/anton/desktop/omnixas
archive run: /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42
features: /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/features
new output root: /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs


## 2. Load and validate exported feature splits

In [2]:
def load_splits(dataset: str, feature_dir: Path = FEATURES_DIR) -> MLSplits:
    parts = {}
    for split_name in SPLITS:
        x_path = feature_dir / f"{dataset}_{split_name}_X.txt"
        y_path = feature_dir / f"{dataset}_{split_name}_y.txt"
        if not x_path.is_file() or not y_path.is_file():
            raise FileNotFoundError(f"Missing feature split for {dataset} {split_name}: {x_path}, {y_path}")
        X = np.atleast_2d(np.loadtxt(x_path, dtype=np.float32))
        y = np.atleast_2d(np.loadtxt(y_path, dtype=np.float32))
        if X.shape[0] != y.shape[0] or X.shape[1] != INPUT_DIM or y.shape[1] != OUTPUT_DIM:
            raise ValueError(f"Invalid {dataset} {split_name}: X={X.shape}, y={y.shape}")
        if not np.isfinite(X).all() or not np.isfinite(y).all():
            raise ValueError(f"Non-finite values in {dataset} {split_name}")
        parts[split_name] = MLData(X=X, y=y)
    return MLSplits(**parts)


def concat_splits(datasets, split_map) -> MLSplits:
    return MLSplits(
        train=MLData(X=np.concatenate([split_map[d].train.X for d in datasets]), y=np.concatenate([split_map[d].train.y for d in datasets])),
        val=MLData(X=np.concatenate([split_map[d].val.X for d in datasets]), y=np.concatenate([split_map[d].val.y for d in datasets])),
        test=MLData(X=np.concatenate([split_map[d].test.X for d in datasets]), y=np.concatenate([split_map[d].test.y for d in datasets])),
    )

splits = {task: load_splits(task) for task in EXPORT_TASKS}
universal_feff = concat_splits(FEFF_TASKS, splits)

display(pd.DataFrame([
    {"dataset": task, "train": len(split.train.X), "val": len(split.val.X), "test": len(split.test.X), "input_dim": split.train.X.shape[1], "output_dim": split.train.y.shape[1]}
    for task, split in splits.items()
]))
print("Universal FEFF:", universal_feff.train.X.shape, universal_feff.val.X.shape, universal_feff.test.X.shape)


,dataset,train,val,test,input_dim,output_dim
0,Ti_FEFF,5140,641,641,64,141
1,V_FEFF,8653,1080,1080,64,141
2,Cr_FEFF,2457,305,305,64,141
3,Mn_FEFF,13644,1704,1704,64,141
4,Fe_FEFF,9657,1205,1205,64,141
5,Co_FEFF,8605,1074,1074,64,141
6,Ni_FEFF,3471,432,432,64,141
7,Cu_FEFF,3340,416,416,64,141
8,Ti_VASP,3030,377,377,64,141
9,Cu_VASP,2584,321,321,64,141


Universal FEFF: (54967, 64) (6857, 64) (6857, 64)


## 3. Training and metric helpers

In [3]:
def best_checkpoint(directory: Path) -> Path | None:
    if not directory.exists():
        return None
    checkpoints = sorted(directory.glob("best*.ckpt"))
    return min(checkpoints, key=checkpoint_score) if checkpoints else None


def ensure_clean_or_complete(directory: Path, label: str) -> None:
    if best_checkpoint(directory) is None and directory.exists() and any(directory.iterdir()):
        raise RuntimeError(f"Incomplete {label} directory exists without a best checkpoint: {directory}")


def write_settings_once(directory: Path, metadata: dict) -> None:
    directory.mkdir(parents=True, exist_ok=True)
    path = directory / "settings.json"
    if path.exists():
        saved = json.loads(path.read_text(encoding="utf-8"))
        if saved != metadata:
            raise ValueError(f"Existing settings mismatch: {path}")
    else:
        path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")


def build_head_regressor(directory: Path, cfg: dict, *, batch: int | None = None, epochs: int | None = None, patience: int | None = None) -> XASBlockRegressor:
    XASBlock.DROPOUT = float(cfg["dropout"])
    scheduler = cfg.get("scheduler", "cosine")
    kwargs = dict(
        directory=str(directory), overwrite_save_dir=False,
        input_dim=INPUT_DIM, output_dim=OUTPUT_DIM, hidden_dims=HEAD_WIDTHS,
        batch_size=int(batch or cfg.get("batch_size", 32)),
        max_epochs=int(epochs or cfg.get("epochs", 800)),
        early_stopping_patience=int(patience or cfg.get("patience", 60)),
        initial_lr=float(cfg["lr"]), min_lr=1e-4,
        use_lr_finder=False, use_early_stopping=True,
        monitor_metric="val_median_mse", shuffle=True, lr_scheduler=scheduler,
    )
    if scheduler in {"cosine", "warmup_cosine"}:
        kwargs.update(cosine_t_max=int(cfg.get("cosine_t", kwargs["max_epochs"])), cosine_eta_min=float(cfg.get("eta_min", 1e-6)))
    if scheduler == "warmup_cosine":
        kwargs.update(warmup_epochs=int(cfg.get("warmup", 10)), warmup_start_factor=float(cfg.get("warmup_start_factor", 0.1)))
    if scheduler == "plateau":
        kwargs.update(plateau_factor=float(cfg.get("plateau_factor", 0.5)), plateau_patience=int(cfg.get("plateau_patience", 5)), plateau_min_lr=float(cfg.get("plateau_min_lr", 1e-6)))
    return XASBlockRegressor(**kwargs)


def score_head_on_tasks(ckpt: Path, datasets, split_name: str) -> list[dict]:
    rows = []
    for task in datasets:
        row = {"dataset": task, "checkpoint": str(ckpt)}
        row.update(eval_ckpt(ckpt, splits[task], split_name))
        rows.append(row)
    return rows


## 4. Train two new UniversalXAS heads on all FEFF exported features

These are feature-only head runs. They do not change the encoder.


In [4]:
def universal_run_dir(cfg: dict) -> Path:
    return NOTEBOOK_RUN_ROOT / "heads" / "universalXAS" / "All_FEFF" / "runs" / cfg["name"]


def train_universal_variant(cfg: dict) -> Path:
    out = universal_run_dir(cfg)
    existing = best_checkpoint(out)
    metadata = {"stage": "UniversalXAS_from_exported_features", "variant": cfg, "source_features": str(FEATURES_DIR), "train_datasets": list(FEFF_TASKS)}
    if existing is not None and RESUME_EXISTING:
        write_settings_once(out, metadata)
        print("SKIP completed UniversalXAS:", existing)
        return existing
    ensure_clean_or_complete(out, f"UniversalXAS {cfg['name']}")
    write_settings_once(out, metadata)
    seed = int(cfg["seed"])
    pl.seed_everything(seed, workers=True); np.random.seed(seed); random.seed(seed); torch.manual_seed(seed)
    print("Training UniversalXAS:", out)
    reg = build_head_regressor(out, cfg, batch=cfg.get("batch_size", 32))
    reg.fit(universal_feff)
    return Path(reg.cfg.fetch_checkpoint("best"))


universal_validation_rows = []
for cfg in UNIVERSAL_VARIANTS:
    ckpt = train_universal_variant(cfg) if RUN_TRAIN_UNIVERSAL else best_checkpoint(universal_run_dir(cfg))
    if ckpt is None:
        raise FileNotFoundError(f"Missing UniversalXAS checkpoint for {cfg['name']}")
    rows = score_head_on_tasks(ckpt, FEFF_TASKS, "val")
    for row in rows:
        row.update({"variant": cfg["name"], "dropout": cfg["dropout"], "lr": cfg["lr"], "scheduler": cfg["scheduler"]})
    universal_validation_rows.extend(rows)

UNIVERSAL_RESULTS_DIR = NOTEBOOK_RUN_ROOT / "analysis" / "new_universal_feff"
UNIVERSAL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
universal_validation_df = pd.DataFrame(universal_validation_rows)
universal_summary_df = (
    universal_validation_df.groupby(["variant", "checkpoint", "dropout", "lr", "scheduler"], as_index=False)["val_eta"]
    .mean().rename(columns={"val_eta": "mean_val_eta"}).sort_values("mean_val_eta", ascending=False)
)
universal_validation_df.to_csv(UNIVERSAL_RESULTS_DIR / "universal_validation_by_dataset.csv", index=False)
universal_summary_df.to_csv(UNIVERSAL_RESULTS_DIR / "universal_validation_summary.csv", index=False)

SELECTED_UNIVERSAL_ROW = universal_summary_df.iloc[0].to_dict()
SELECTED_UNIVERSAL_CKPT = Path(SELECTED_UNIVERSAL_ROW["checkpoint"])
print("Selected UniversalXAS by mean validation eta:", SELECTED_UNIVERSAL_CKPT)
display(universal_summary_df.style.format(precision=4))
display(universal_validation_df.sort_values(["dataset", "val_eta"], ascending=[True, False]).style.format(precision=4))


Seed set to 43


Training UniversalXAS: /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_cosT800_lr7e-4_do025_seed43


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_cosT800_lr7e-4_do025_seed43 exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type     | Params | Mode  | FLOPs
---------------------------------------------------
0 | loss  | MSELoss  | 0      | train | 0    
1 | model | XASBlock | 639 K  | train | 0    
---------------------------------------------------
639 K     Trainable params
0         Non-trainable pa

Sanity Checking: |                                                                                | 0/? [00:00…

/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.
/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.


...

Validation: |                                                                                     | 0/? [00:00…

2026-07-28 10:17:51.058 | INFO     | omnixas.model.xasblock_regressor:fit:246 - Best models saved at /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_plateau_lr5e-4_do010_seed44/best-model-epoch=279-val_median_mse=0.0022.ckpt
2026-07-28 10:17:51.058 | INFO     | omnixas.model.xasblock_regressor:fit:247 - Best validation loss: 0.004922823514789343


Selected UniversalXAS by mean validation eta: /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_plateau_lr5e-4_do010_seed44/best-model-epoch=279-val_median_mse=0.0022.ckpt


,variant,checkpoint,dropout,lr,scheduler,mean_val_eta
1,feff_plateau_lr5e-4_do010_seed44,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_plateau_lr5e-4_do010_seed44/best-model-epoch=279-val_median_mse=0.0022.ckpt,0.1000,0.0005,plateau,15.8569
0,feff_cosT800_lr7e-4_do025_seed43,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_cosT800_lr7e-4_do025_seed43/best-model-epoch=191-val_median_mse=0.0025.ckpt,0.2500,0.0007,cosine,14.3401


,dataset,checkpoint,val_median_mse,val_baseline_median_mse,val_eta,variant,dropout,lr,scheduler
13,Co_FEFF,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_plateau_lr5e-4_do010_seed44/best-model-epoch=279-val_median_mse=0.0022.ckpt,0.0011,0.0309,27.5355,feff_plateau_lr5e-4_do010_seed44,0.1000,0.0005,plateau
5,Co_FEFF,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_cosT800_lr7e-4_do025_seed43/best-model-epoch=191-val_median_mse=0.0025.ckpt,0.0013,0.0309,23.5124,feff_cosT800_lr7e-4_do025_seed43,0.2500,0.0007,cosine
10,Cr_FEFF,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_plateau_lr5e-4_do010_seed44/best-model-epoch=279-val_median_mse=0.0022.ckpt,0.0039,0.0561,14.2044,feff_plateau_lr5e-4_do010_seed44,0.1000,0.0005,plateau
2,Cr_FEFF,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_cosT800_lr7e-4_do025_seed43/best-model-epoch=191-val_median_mse=0.0025.ckpt,0.0041,0.0561,13.8205,feff_cosT800_lr7e-4_do025_seed43,0.2500,0.0007,cosine
15,Cu_FEFF,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_plateau_lr5e-4_do010_seed44/best-model-epoch=279-val_median_mse=0.0022.ckpt,0.0021,0.0125,5.9775,feff_plateau_lr5e-4_do010_seed44,0.1000,0.0005,plateau
7,Cu_FEFF,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_cosT800_lr7e-4_do025_seed43/best-model-epoch=191-val_median_mse=0.0025.ckpt,0.0021,0.0125,5.8529,feff_cosT800_lr7e-4_do025_seed43,0.2500,0.0007,cosine
12,Fe_FEFF,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_plateau_lr5e-4_do010_seed44/best-model-epoch=279-val_median_mse=0.0022.ckpt,0.0020,0.0291,14.1925,feff_plateau_lr5e-4_do010_seed44,0.1000,0.0005,plateau
4,Fe_FEFF,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_cosT800_lr7e-4_do025_seed43/best-model-epoch=191-val_median_mse=0.0025.ckpt,0.0022,0.0291,13.1888,feff_cosT800_lr7e-4_do025_seed43,0.2500,0.0007,cosine
11,Mn_FEFF,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_plateau_lr5e-4_do010_seed44/best-model-epoch=279-val_median_mse=0.0022.ckpt,0.0018,0.0473,26.6693,feff_plateau_lr5e-4_do010_seed44,0.1000,0.0005,plateau
3,Mn_FEFF,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_cosT800_lr7e-4_do025_seed43/best-model-epoch=191-val_median_mse=0.0025.ckpt,0.0021,0.0473,22.0454,feff_cosT800_lr7e-4_do025_seed43,0.2500,0.0007,cosine


## 5. Optional final test evaluation for selected UniversalXAS

Leave this off until you are done changing the UniversalXAS candidate set.


In [5]:
if not RUN_FINAL_TEST:
    print("Test evaluation skipped. Set RUN_FINAL_TEST=True only after validation selection is final.")
else:
    artifact = UNIVERSAL_RESULTS_DIR / "selected_universal_test_by_dataset.csv"
    if artifact.exists():
        raise FileExistsError(f"Test artifact already exists: {artifact}")
    test_df = pd.DataFrame(score_head_on_tasks(SELECTED_UNIVERSAL_CKPT, FEFF_TASKS, "test"))
    test_df.to_csv(artifact, index=False)
    display(test_df.style.format(precision=4))


,dataset,checkpoint,test_median_mse,test_baseline_median_mse,test_eta
0,Ti_FEFF,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_plateau_lr5e-4_do010_seed44/best-model-epoch=279-val_median_mse=0.0022.ckpt,0.0045,0.0486,10.7756
1,V_FEFF,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_plateau_lr5e-4_do010_seed44/best-model-epoch=279-val_median_mse=0.0022.ckpt,0.0045,0.0507,11.1704
2,Cr_FEFF,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_plateau_lr5e-4_do010_seed44/best-model-epoch=279-val_median_mse=0.0022.ckpt,0.0032,0.0563,17.3858
3,Mn_FEFF,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_plateau_lr5e-4_do010_seed44/best-model-epoch=279-val_median_mse=0.0022.ckpt,0.0018,0.0456,25.5042
4,Fe_FEFF,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_plateau_lr5e-4_do010_seed44/best-model-epoch=279-val_median_mse=0.0022.ckpt,0.0021,0.0288,13.8655
5,Co_FEFF,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_plateau_lr5e-4_do010_seed44/best-model-epoch=279-val_median_mse=0.0022.ckpt,0.0012,0.0300,25.5464
6,Ni_FEFF,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_plateau_lr5e-4_do010_seed44/best-model-epoch=279-val_median_mse=0.0022.ckpt,0.0017,0.0253,15.2515
7,Cu_FEFF,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_plateau_lr5e-4_do010_seed44/best-model-epoch=279-val_median_mse=0.0022.ckpt,0.0021,0.0128,6.0692


## 6. Fine-tune the selected UniversalXAS on every FEFF element

This creates per-element Tuned-UniversalXAS heads from the validation-selected UniversalXAS source.


In [6]:
def tuned_feff_run_dir(task: str, cfg: dict, source_ckpt: Path) -> Path:
    return NOTEBOOK_RUN_ROOT / "heads" / "tunedUniversalXAS" / task / "runs" / f"{cfg['name']}_from_{source_ckpt.parent.name}"


def tune_feff_head(task: str, source_ckpt: Path, cfg: dict) -> Path:
    out = tuned_feff_run_dir(task, cfg, source_ckpt)
    existing = best_checkpoint(out)
    metadata = {"stage": "Tuned-UniversalXAS_FEFF", "dataset": task, "variant": cfg, "source_checkpoint": str(source_ckpt), "source_features": str(FEATURES_DIR)}
    if existing is not None and RESUME_EXISTING:
        write_settings_once(out, metadata)
        print("SKIP completed tuned FEFF:", existing)
        return existing
    ensure_clean_or_complete(out, f"tuned {task} {cfg['name']}")
    write_settings_once(out, metadata)
    seed = int(cfg["seed"])
    pl.seed_everything(seed, workers=True); np.random.seed(seed); random.seed(seed); torch.manual_seed(seed)
    reg = build_head_regressor(source_ckpt.parent, cfg, batch=BATCH[task], epochs=cfg["epochs"], patience=cfg["patience"])
    reg.load("best")
    reg.cfg.directory = str(out)
    print("Fine-tuning", task, cfg["name"], "from", source_ckpt)
    reg.fit(splits[task])
    return Path(reg.cfg.fetch_checkpoint("best"))


feff_candidate_rows, feff_selected_rows = [], []
if not RUN_TUNE_FEFF:
    print("FEFF tuning skipped.")
else:
    for task in FEFF_TASKS:
        task_rows = []
        for cfg in FEFF_TUNING_SWEEP:
            ckpt = tune_feff_head(task, SELECTED_UNIVERSAL_CKPT, cfg)
            row = {"dataset": task, "setting": cfg["name"], "dropout": cfg["dropout"], "lr": cfg["lr"], "scheduler": cfg["scheduler"], "checkpoint": str(ckpt), "source_universal_checkpoint": str(SELECTED_UNIVERSAL_CKPT), "val_loss_score": checkpoint_score(ckpt)}
            row.update(eval_ckpt(ckpt, splits[task], "val"))
            feff_candidate_rows.append(row); task_rows.append(row)
            print(f"{task} {cfg['name']}: val_eta={row['val_eta']:.4f}")
        selected = dict(max(task_rows, key=lambda r: r["val_eta"]))
        if RUN_FINAL_TEST:
            selected.update(eval_ckpt(Path(selected["checkpoint"]), splits[task], "test"))
        feff_selected_rows.append(selected)

    feff_candidates_df = pd.DataFrame(feff_candidate_rows).sort_values(["dataset", "val_eta"], ascending=[True, False])
    feff_selected_df = pd.DataFrame(feff_selected_rows).sort_values("dataset")
    feff_candidates_df.to_csv(UNIVERSAL_RESULTS_DIR / "tuned_feff_validation_candidates.csv", index=False)
    feff_selected_df.to_csv(UNIVERSAL_RESULTS_DIR / "tuned_feff_selected_by_validation.csv", index=False)
    display(feff_candidates_df.style.format(precision=4))
    display(feff_selected_df.style.format(precision=4))


Seed set to 143
2026-07-28 10:17:54.485 | INFO     | omnixas.model.xasblock_regressor:load:265 - Loading model from /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_plateau_lr5e-4_do010_seed44/best-model-epoch=279-val_median_mse=0.0022.ckpt
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Fine-tuning Ti_FEFF cosT250_lr1e-4_do0p0 from /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_plateau_lr5e-4_do010_seed44/best-model-epoch=279-val_median_mse=0.0022.ckpt


/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/tunedUniversalXAS/Ti_FEFF/runs/cosT250_lr1e-4_do0p0_from_feff_plateau_lr5e-4_do010_seed44 exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type     | Params | Mode  | FLOPs
---------------------------------------------------
0 | loss  | MSELoss  | 0      | train | 0    
1 | model | XASBlock | 639 K  | train | 0    
---------------------------------------------------
639 K     Trainable params
0         Non-trainable params
639 K     Total params
2.557     Total estimated model params size (MB)
16        Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |                                                                                | 0/? [00:00…

/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.
/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.


...

Validation: |                                                                                     | 0/? [00:00…

2026-07-28 11:21:44.601 | INFO     | omnixas.model.xasblock_regressor:fit:246 - Best models saved at /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/tunedUniversalXAS/Cu_FEFF/runs/cosT500_lr3e-4_do0p1_from_feff_plateau_lr5e-4_do010_seed44/best-model-epoch=97-val_median_mse=0.0018.ckpt
2026-07-28 11:21:44.601 | INFO     | omnixas.model.xasblock_regressor:fit:247 - Best validation loss: 0.0027084648609161377


Cu_FEFF cosT500_lr3e-4_do0p1: val_eta=6.9965


,dataset,setting,dropout,lr,scheduler,checkpoint,source_universal_checkpoint,val_loss_score,val_median_mse,val_baseline_median_mse,val_eta
17,Co_FEFF,cosT500_lr3e-4_do0p1,0.1000,0.0003,cosine,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/tunedUniversalXAS/Co_FEFF/runs/cosT500_lr3e-4_do0p1_from_feff_plateau_lr5e-4_do010_seed44/best-model-epoch=321-val_median_mse=0.0010.ckpt,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_plateau_lr5e-4_do010_seed44/best-model-epoch=279-val_median_mse=0.0022.ckpt,0.0010,0.0010,0.0309,31.8323
16,Co_FEFF,cosT250_lr1e-4_do0p1,0.1000,0.0001,cosine,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/tunedUniversalXAS/Co_FEFF/runs/cosT250_lr1e-4_do0p1_from_feff_plateau_lr5e-4_do010_seed44/best-model-epoch=143-val_median_mse=0.0010.ckpt,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_plateau_lr5e-4_do010_seed44/best-model-epoch=279-val_median_mse=0.0022.ckpt,0.0010,0.0010,0.0309,29.8395
15,Co_FEFF,cosT250_lr1e-4_do0p0,0.0000,0.0001,cosine,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/tunedUniversalXAS/Co_FEFF/runs/cosT250_lr1e-4_do0p0_from_feff_plateau_lr5e-4_do010_seed44/best-model-epoch=119-val_median_mse=0.0010.ckpt,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_plateau_lr5e-4_do010_seed44/best-model-epoch=279-val_median_mse=0.0022.ckpt,0.0010,0.0010,0.0309,29.5834
8,Cr_FEFF,cosT500_lr3e-4_do0p1,0.1000,0.0003,cosine,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/tunedUniversalXAS/Cr_FEFF/runs/cosT500_lr3e-4_do0p1_from_feff_plateau_lr5e-4_do010_seed44/best-model-epoch=103-val_median_mse=0.0037.ckpt,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_plateau_lr5e-4_do010_seed44/best-model-epoch=279-val_median_mse=0.0022.ckpt,0.0037,0.0037,0.0561,15.1441
7,Cr_FEFF,cosT250_lr1e-4_do0p1,0.1000,0.0001,cosine,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/tunedUniversalXAS/Cr_FEFF/runs/cosT250_lr1e-4_do0p1_from_feff_plateau_lr5e-4_do010_seed44/best-model-epoch=151-val_median_mse=0.0038.ckpt,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_plateau_lr5e-4_do010_seed44/best-model-epoch=279-val_median_mse=0.0022.ckpt,0.0038,0.0038,0.0561,14.7375
6,Cr_FEFF,cosT250_lr1e-4_do0p0,0.0000,0.0001,cosine,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/tunedUniversalXAS/Cr_FEFF/runs/cosT250_lr1e-4_do0p0_from_feff_plateau_lr5e-4_do010_seed44/best-model-epoch=91-val_median_mse=0.0039.ckpt,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_plateau_lr5e-4_do010_seed44/best-model-epoch=279-val_median_mse=0.0022.ckpt,0.0039,0.0039,0.0561,14.2428
23,Cu_FEFF,cosT500_lr3e-4_do0p1,0.1000,0.0003,cosine,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/tunedUniversalXAS/Cu_FEFF/runs/cosT500_lr3e-4_do0p1_from_feff_plateau_lr5e-4_do010_seed44/best-model-epoch=97-val_median_mse=0.0018.ckpt,/mnt/c/Users/anton/desktop/fulltrainingcopy

,dataset,setting,dropout,lr,scheduler,checkpoint,source_universal_checkpoint,val_loss_score,val_median_mse,val_baseline_median_mse,val_eta,test_median_mse,test_baseline_median_mse,test_eta
5,Co_FEFF,cosT500_lr3e-4_do0p1,0.1000,0.0003,cosine,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/tunedUniversalXAS/Co_FEFF/runs/cosT500_lr3e-4_do0p1_from_feff_plateau_lr5e-4_do010_seed44/best-model-epoch=321-val_median_mse=0.0010.ckpt,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_plateau_lr5e-4_do010_seed44/best-model-epoch=279-val_median_mse=0.0022.ckpt,0.0010,0.0010,0.0309,31.8323,0.0010,0.0300,30.0408
2,Cr_FEFF,cosT500_lr3e-4_do0p1,0.1000,0.0003,cosine,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/tunedUniversalXAS/Cr_FEFF/runs/cosT500_lr3e-4_do0p1_from_feff_plateau_lr5e-4_do010_seed44/best-model-epoch=103-val_median_mse=0.0037.ckpt,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_plateau_lr5e-4_do010_seed44/best-model-epoch=279-val_median_mse=0.0022.ckpt,0.0037,0.0037,0.0561,15.1441,0.0031,0.0563,18.1315
7,Cu_FEFF,cosT500_lr3e-4_do0p1,0.1000,0.0003,cosine,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/tunedUniversalXAS/Cu_FEFF/runs/cosT500_lr3e-4_do0p1_from_feff_plateau_lr5e-4_do010_seed44/best-model-epoch=97-val_median_mse=0.0018.ckpt,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_plateau_lr5e-4_do010_seed44/best-model-epoch=279-val_median_mse=0.0022.ckpt,0.0018,0.0018,0.0125,6.9965,0.0019,0.0128,6.6978
4,Fe_FEFF,cosT500_lr3e-4_do0p1,0.1000,0.0003,cosine,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/tunedUniversalXAS/Fe_FEFF/runs/cosT500_lr3e-4_do0p1_from_feff_plateau_lr5e-4_do010_seed44/best-model-epoch=187-val_median_mse=0.0019.ckpt,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_plateau_lr5e-4_do010_seed44/best-model-epoch=279-val_median_mse=0.0022.ckpt,0.0019,0.0019,0.0291,15.3704,0.0019,0.0288,15.0637
3,Mn_FEFF,cosT500_lr3e-4_do0p1,0.1000,0.0003,cosine,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/tunedUniversalXAS/Mn_FEFF/runs/cosT500_lr3e-4_do0p1_from_feff_plateau_lr5e-4_do010_seed44/best-model-epoch=351-val_median_mse=0.0013.ckpt,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_plateau_lr5e-4_do010_seed44/best-model-epoch=279-val_median_mse=0.0022.ckpt,0.0013,0.0013,0.0473,36.3367,0.0013,0.0456,33.9660
6,Ni_FEFF,cosT500_lr3e-4_do0p1,0.1000,0.0003,cosine,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/tunedUniversalXAS/Ni_FEFF/runs/cosT500_lr3e-4_do0p1_from_feff_plateau_lr5e-4_do010_seed44/best-model-epoch=145-val_median_mse=0.0015.ckpt,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_plateau_lr5e-4_do010_seed44/best-model-epoch=279-val_median_mse=0.0022.ckpt,0.0015,0.0015,0.0259,17.5797,0.0016,0.0253,16.2418
0,Ti_FEFF,cosT500_lr3e-4_do0p1,0.1000,0.0003,cosine,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/tune

## 7. VASP stage config: fine-tune encoder + provided UniversalXAS head

This is intentionally a separate long-running stage because it needs raw structures. The UniversalXAS head is loaded from `VASP_SUPERVISOR_HEAD_CKPT` and remains trainable together with the encoder.


In [7]:
RUN_VASP_ENCODER_FINETUNE = True  # long raw-structure GPU stage
RUN_EXPORT_VASP_TUNED_FEATURES = True
RUN_TUNE_UNIVERSAL_AFTER_VASP_ENCODER = True

RAW_ROOT = Path(os.environ.get("OMNIXAS_DATA_ROOT", REPO_ROOT.parent / "OmniXAS_data")).expanduser() / "materialscloud_omnixas_raw" / "extracted"
VASP_SOURCE_ENCODER_CKPT = ARCHIVE_RUN / "best_e2e_encoder_universal.ckpt"


def default_universal_head_ckpt() -> Path:
    if "SELECTED_UNIVERSAL_CKPT" in globals() and Path(SELECTED_UNIVERSAL_CKPT).is_file():
        return Path(SELECTED_UNIVERSAL_CKPT)
    candidates = sorted((ARCHIVE_RUN / "heads" / "universalXAS" / "All_FEFF" / "runs").glob("*/best*.ckpt"))
    if not candidates:
        raise FileNotFoundError("No UniversalXAS head checkpoint found. Run sections 4-6 or set VASP_SUPERVISOR_HEAD_CKPT manually.")
    return min(candidates, key=checkpoint_score)

VASP_SUPERVISOR_HEAD_CKPT = default_universal_head_ckpt()
VASP_E2E_CFG = {"name": "vasp_encoder_plus_unfrozen_universal_head_seed243", "seed": 243, "gnn_dropout": 0.10, "head_dropout": 0.10, "encoder_lr": 2e-5, "head_lr": 2e-4, "weight_decay": 1e-5, "epochs": 120, "patience": 18, "batch_size": ENCODER_BATCH, "num_workers": 4}
VASP_E2E_RUN_DIR = NOTEBOOK_RUN_ROOT / "vasp_encoder_plus_universal_head" / VASP_E2E_CFG["name"]
VASP_TUNED_FEATURES_DIR = VASP_E2E_RUN_DIR / "features"

print("raw root:", RAW_ROOT)
print("source encoder:", VASP_SOURCE_ENCODER_CKPT)
print("trainable supervisor UniversalXAS head:", VASP_SUPERVISOR_HEAD_CKPT)
print("vasp run dir:", VASP_E2E_RUN_DIR)


raw root: /mnt/c/Users/anton/desktop/OmniXAS_data/materialscloud_omnixas_raw/extracted
source encoder: /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/best_e2e_encoder_universal.ckpt
trainable supervisor UniversalXAS head: /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_plateau_lr5e-4_do010_seed44/best-model-epoch=279-val_median_mse=0.0022.ckpt
vasp run dir: /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/vasp_encoder_plus_universal_head/vasp_encoder_plus_unfrozen_universal_head_seed243


## 8. VASP end-to-end helpers

The key invariant is enforced in `make_vasp_model`: the UniversalXAS head is **not frozen**.


In [8]:
class TaskCollateGraphs(CollateGraphs):
    def __init__(self, encoder, tasks):
        super().__init__(encoder)
        self.task_idx = {task: i for i, task in enumerate(tasks)}


def state_dict_from_checkpoint(ckpt: Path) -> dict[str, torch.Tensor]:
    state = torch_load(ckpt).get("state_dict")
    if state is None:
        raise ValueError(f"Missing state_dict in {ckpt}")
    return state


def xasblock_state_from_checkpoint(ckpt: Path) -> dict[str, torch.Tensor]:
    state = state_dict_from_checkpoint(ckpt)
    if any(k.startswith("model.head.") for k in state):
        return {k.removeprefix("model.head."): v for k, v in state.items() if k.startswith("model.head.")}
    if any(k.startswith("model.") for k in state):
        return {k.removeprefix("model."): v for k, v in state.items() if k.startswith("model.")}
    raise ValueError(f"Could not find an XASBlock state in {ckpt}")


def encoder_state_from_e2e_checkpoint(ckpt: Path) -> dict[str, torch.Tensor]:
    state = state_dict_from_checkpoint(ckpt)
    encoder_state = {k.removeprefix("model.encoder."): v for k, v in state.items() if k.startswith("model.encoder.")}
    if not encoder_state:
        raise ValueError(f"Could not find model.encoder.* weights in {ckpt}")
    return encoder_state


def make_vasp_model(encoder_ckpt: Path, head_ckpt: Path, cfg: dict) -> EndToEndUniversal:
    model = EndToEndUniversal(cfg["gnn_dropout"], cfg["head_dropout"])
    model.encoder.load_state_dict(encoder_state_from_e2e_checkpoint(encoder_ckpt), strict=True)
    model.head.load_state_dict(xasblock_state_from_checkpoint(head_ckpt), strict=True)
    for p in model.encoder.parameters(): p.requires_grad = True
    for p in model.head.parameters(): p.requires_grad = True
    if not all(p.requires_grad for p in model.head.parameters()):
        raise RuntimeError("UniversalXAS head is frozen; this stage requires it to be trainable.")
    return model


def task_baselines(tasks):
    data = REPO_ROOT / "tutorial_omnixas" / "ml_data"
    train_base, val_base, counts = [], [], []
    for task in tasks:
        train_y = np.atleast_2d(np.loadtxt(data / f"{task}_train_y.txt", dtype=np.float32))
        counts.append(len(train_y))
        for split_name, out in (("train", train_base), ("val", val_base)):
            y = np.atleast_2d(np.loadtxt(data / f"{task}_{split_name}_y.txt", dtype=np.float32))
            baseline = np.repeat(train_y.mean(axis=0, keepdims=True), len(y), axis=0)
            out.append(float(np.median(np.mean((y - baseline) ** 2, axis=1))))
    counts = torch.tensor(counts, dtype=torch.float32)
    weights = counts.sum() / (len(tasks) * counts)
    return torch.tensor(train_base), torch.tensor(val_base), weights


## 9. Run VASP encoder + unfrozen UniversalXAS-head fine-tuning

In [9]:
def existing_vasp_e2e_best() -> Path | None:
    stable = VASP_E2E_RUN_DIR / "best_vasp_encoder_plus_universal_head.ckpt"
    if stable.is_file():
        return stable
    ckpt = best_checkpoint(VASP_E2E_RUN_DIR / "checkpoints")
    if ckpt is not None:
        stable.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(ckpt, stable)
        return stable
    return None


VASP_E2E_BEST_CKPT = existing_vasp_e2e_best()
if not RUN_VASP_ENCODER_FINETUNE:
    print("VASP encoder fine-tuning skipped. Set RUN_VASP_ENCODER_FINETUNE=True to run it.")
    print("Existing best:", VASP_E2E_BEST_CKPT)
else:
    if VASP_E2E_BEST_CKPT is not None and RESUME_EXISTING:
        print("SKIP completed VASP E2E run:", VASP_E2E_BEST_CKPT)
    else:
        if VASP_E2E_RUN_DIR.exists() and any(VASP_E2E_RUN_DIR.iterdir()):
            raise RuntimeError(f"Incomplete VASP E2E run exists: {VASP_E2E_RUN_DIR}")
        if not VASP_SOURCE_ENCODER_CKPT.is_file(): raise FileNotFoundError(VASP_SOURCE_ENCODER_CKPT)
        if not VASP_SUPERVISOR_HEAD_CKPT.is_file(): raise FileNotFoundError(VASP_SUPERVISOR_HEAD_CKPT)
        patch_matgl_gpu_constants()
        validate_raw_structures(REPO_ROOT, RAW_ROOT, list(VASP_TASKS))
        VASP_E2E_RUN_DIR.mkdir(parents=True, exist_ok=True)
        (VASP_E2E_RUN_DIR / "settings.json").write_text(json.dumps({"stage": "VASP_encoder_plus_unfrozen_UniversalXAS_head", "cfg": VASP_E2E_CFG, "source_encoder_checkpoint": str(VASP_SOURCE_ENCODER_CKPT), "supervisor_head_checkpoint": str(VASP_SUPERVISOR_HEAD_CKPT), "tasks": list(VASP_TASKS), "raw_root": str(RAW_ROOT)}, indent=2), encoding="utf-8")
        seed = VASP_E2E_CFG["seed"]
        pl.seed_everything(seed, workers=True); np.random.seed(seed); random.seed(seed); torch.manual_seed(seed)
        model = make_vasp_model(VASP_SOURCE_ENCODER_CKPT, VASP_SUPERVISOR_HEAD_CKPT, VASP_E2E_CFG)
        print("trainable encoder params:", sum(p.numel() for p in model.encoder.parameters() if p.requires_grad))
        print("trainable head params:", sum(p.numel() for p in model.head.parameters() if p.requires_grad))
        collate = TaskCollateGraphs(model.encoder, VASP_TASKS)
        train_loader = DataLoader(FEFFDataset(REPO_ROOT, RAW_ROOT, list(VASP_TASKS), "train"), batch_size=VASP_E2E_CFG["batch_size"], shuffle=True, collate_fn=collate, num_workers=VASP_E2E_CFG["num_workers"], drop_last=True)
        val_loader = DataLoader(FEFFDataset(REPO_ROOT, RAW_ROOT, list(VASP_TASKS), "val"), batch_size=VASP_E2E_CFG["batch_size"], shuffle=False, collate_fn=collate, num_workers=VASP_E2E_CFG["num_workers"])
        train_base, val_base, weights = task_baselines(VASP_TASKS)
        lit = LitEndToEnd(model, list(VASP_TASKS), train_base, val_base, weights, VASP_E2E_CFG["encoder_lr"], VASP_E2E_CFG["head_lr"], VASP_E2E_CFG["weight_decay"], VASP_E2E_CFG["epochs"])
        ckpt_cb = ModelCheckpoint(dirpath=VASP_E2E_RUN_DIR / "checkpoints", filename="best-{epoch:03d}-{val_balanced_rel_mse:.5f}", monitor="val_balanced_rel_mse", mode="min", save_top_k=1, save_last=True)
        trainer = pl.Trainer(max_epochs=VASP_E2E_CFG["epochs"], accelerator="auto", devices=1, callbacks=[EarlyStopping(monitor="val_balanced_rel_mse", patience=VASP_E2E_CFG["patience"], mode="min"), ckpt_cb], logger=CSVLogger(str(VASP_E2E_RUN_DIR), name="logs"), log_every_n_steps=1)
        trainer.fit(lit, train_loader, val_loader)
        if not ckpt_cb.best_model_path: raise RuntimeError("VASP E2E training finished without a best checkpoint")
        VASP_E2E_BEST_CKPT = VASP_E2E_RUN_DIR / "best_vasp_encoder_plus_universal_head.ckpt"
        shutil.copy2(ckpt_cb.best_model_path, VASP_E2E_BEST_CKPT)
        print("best VASP E2E checkpoint:", VASP_E2E_BEST_CKPT)


Seed set to 243


trainable encoder params: 263067
trainable head params: 639341


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type              | Params | Mode  | FLOPs
------------------------------------------------------------
0 | model | EndToEndUniversal | 902 K  | train | 0    
------------------------------------------------------------
902 K     Trainable params
0         Non-trainable params
902 K     Total params
3.610     Total estimated model params size (MB)
170       Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |                                                                                | 0/? [00:00…

Training: |                                                                                       | 0/? [00:00…

...

Validation: |                                                                                     | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

best VASP E2E checkpoint: /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/vasp_encoder_plus_universal_head/vasp_encoder_plus_unfrozen_universal_head_seed243/best_vasp_encoder_plus_universal_head.ckpt


## 10. Export VASP features from the VASP-tuned encoder

In [10]:
def load_vasp_finetuned_model(ckpt: Path) -> EndToEndUniversal:
    model = make_vasp_model(VASP_SOURCE_ENCODER_CKPT, VASP_SUPERVISOR_HEAD_CKPT, VASP_E2E_CFG)
    state = state_dict_from_checkpoint(ckpt)
    model.load_state_dict({k.removeprefix("model."): v for k, v in state.items() if k.startswith("model.")}, strict=True)
    return model


@torch.inference_mode()
def export_vasp_features_from_encoder(ckpt: Path, output_dir: Path) -> None:
    patch_matgl_gpu_constants()
    validate_raw_structures(REPO_ROOT, RAW_ROOT, list(VASP_TASKS))
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model = load_vasp_finetuned_model(ckpt).to(device).eval()
    collate = TaskCollateGraphs(model.encoder, VASP_TASKS)
    output_dir.mkdir(parents=True, exist_ok=True)
    for task in VASP_TASKS:
        for split_name in SPLITS:
            x_path = output_dir / f"{task}_{split_name}_X.txt"
            y_path = output_dir / f"{task}_{split_name}_y.txt"
            if x_path.exists() and y_path.exists() and RESUME_EXISTING:
                print("SKIP exported", task, split_name)
                continue
            loader = DataLoader(FEFFDataset(REPO_ROOT, RAW_ROOT, [task], split_name), batch_size=VASP_E2E_CFG["batch_size"], shuffle=False, collate_fn=collate, num_workers=VASP_E2E_CFG["num_workers"])
            Xs, ys = [], []
            for batch in loader:
                sizes = batch["graph"].batch_num_nodes()
                expected_sites = torch.cat([sizes.new_zeros(1), sizes.cumsum(0)[:-1]])
                if not torch.equal(batch["site"], expected_sites):
                    raise RuntimeError(f"VASP absorbers are not node 0 for {task} {split_name}")
                z = model.encoder(batch["graph"].to(device), batch["site"].to(device))
                Xs.append((z * FEATURE_SCALE).cpu().numpy())
                ys.append(batch["y"].numpy())
            X = np.concatenate(Xs); y = np.concatenate(ys)
            if X.shape != (len(y), INPUT_DIM) or y.shape[1] != OUTPUT_DIM:
                raise ValueError(f"Invalid exported features for {task} {split_name}: X={X.shape} y={y.shape}")
            np.savetxt(x_path, X); np.savetxt(y_path, y)
            print("exported", task, split_name, X.shape, y.shape)


if not RUN_EXPORT_VASP_TUNED_FEATURES:
    print("VASP feature export skipped. Set RUN_EXPORT_VASP_TUNED_FEATURES=True after the encoder stage completes.")
    print("feature dir:", VASP_TUNED_FEATURES_DIR)
elif VASP_E2E_BEST_CKPT is None:
    raise FileNotFoundError("No VASP E2E best checkpoint available to export from.")
else:
    export_vasp_features_from_encoder(Path(VASP_E2E_BEST_CKPT), VASP_TUNED_FEATURES_DIR)


exported Ti_VASP train (3030, 64) (3030, 141)
exported Ti_VASP val (377, 64) (377, 141)
exported Ti_VASP test (377, 64) (377, 141)
exported Cu_VASP train (2584, 64) (2584, 141)
exported Cu_VASP val (321, 64) (321, 141)
exported Cu_VASP test (321, 64) (321, 141)


## 11. Fine-tune UniversalXAS after the encoder has been tuned on VASP

This freezes the new VASP-tuned encoder feature space and fine-tunes a UniversalXAS head on combined `Ti_VASP + Cu_VASP` features.


In [11]:
VASP_HEAD_TUNE_SWEEP = [
    {"name": "all_vasp_cosT600_lr4e-4_do025_seed343", "seed": 343, "dropout": 0.25, "lr": 4e-4, "scheduler": "cosine", "epochs": 800, "patience": 40, "batch_size": 32, "cosine_t": 600},
    {"name": "all_vasp_plateau_lr3e-4_do015_seed344", "seed": 344, "dropout": 0.15, "lr": 3e-4, "scheduler": "plateau", "epochs": 800, "patience": 40, "batch_size": 32, "plateau_patience": 8},
]


def vasp_after_encoder_run_dir(cfg: dict) -> Path:
    return VASP_E2E_RUN_DIR / "heads" / "universalXAS" / "All_VASP_after_encoder" / "runs" / cfg["name"]


def tune_universal_after_vasp_encoder(cfg: dict, vasp_feature_splits: dict[str, MLSplits]) -> Path:
    out = vasp_after_encoder_run_dir(cfg)
    existing = best_checkpoint(out)
    metadata = {"stage": "UniversalXAS_after_VASP_encoder_finetune", "variant": cfg, "source_checkpoint": str(VASP_SUPERVISOR_HEAD_CKPT), "source_vasp_e2e_checkpoint": str(VASP_E2E_BEST_CKPT), "source_features": str(VASP_TUNED_FEATURES_DIR), "train_datasets": list(VASP_TASKS)}
    if existing is not None and RESUME_EXISTING:
        write_settings_once(out, metadata)
        print("SKIP completed post-encoder VASP UniversalXAS:", existing)
        return existing
    ensure_clean_or_complete(out, f"post-encoder VASP UniversalXAS {cfg['name']}")
    write_settings_once(out, metadata)
    seed = int(cfg["seed"])
    pl.seed_everything(seed, workers=True); np.random.seed(seed); random.seed(seed); torch.manual_seed(seed)
    all_vasp = concat_splits(VASP_TASKS, vasp_feature_splits)
    reg = build_head_regressor(VASP_SUPERVISOR_HEAD_CKPT.parent, cfg, batch=cfg.get("batch_size", 32), epochs=cfg["epochs"], patience=cfg["patience"])
    reg.load("best")
    reg.cfg.directory = str(out)
    print("Fine-tuning UniversalXAS on VASP-tuned encoder features:", out)
    reg.fit(all_vasp)
    return Path(reg.cfg.fetch_checkpoint("best"))


if not RUN_TUNE_UNIVERSAL_AFTER_VASP_ENCODER:
    print("Post-encoder UniversalXAS VASP fine-tuning skipped. Set RUN_TUNE_UNIVERSAL_AFTER_VASP_ENCODER=True after exporting features.")
else:
    vasp_feature_splits = {task: load_splits(task, VASP_TUNED_FEATURES_DIR) for task in VASP_TASKS}
    vasp_candidate_rows = []
    for cfg in VASP_HEAD_TUNE_SWEEP:
        ckpt = tune_universal_after_vasp_encoder(cfg, vasp_feature_splits)
        for task in VASP_TASKS:
            row = {"dataset": task, "setting": cfg["name"], "dropout": cfg["dropout"], "lr": cfg["lr"], "scheduler": cfg["scheduler"], "checkpoint": str(ckpt), "source_head_checkpoint": str(VASP_SUPERVISOR_HEAD_CKPT), "source_vasp_e2e_checkpoint": str(VASP_E2E_BEST_CKPT)}
            row.update(eval_ckpt(ckpt, vasp_feature_splits[task], "val"))
            if RUN_FINAL_TEST:
                row.update(eval_ckpt(ckpt, vasp_feature_splits[task], "test"))
            vasp_candidate_rows.append(row)
    vasp_candidate_df = pd.DataFrame(vasp_candidate_rows)
    vasp_summary_df = vasp_candidate_df.groupby(["setting", "checkpoint", "dropout", "lr", "scheduler"], as_index=False)["val_eta"].mean().rename(columns={"val_eta": "mean_val_eta"}).sort_values("mean_val_eta", ascending=False)
    out_dir = VASP_E2E_RUN_DIR / "analysis"; out_dir.mkdir(parents=True, exist_ok=True)
    vasp_candidate_df.to_csv(out_dir / "post_encoder_universal_vasp_validation_by_dataset.csv", index=False)
    vasp_summary_df.to_csv(out_dir / "post_encoder_universal_vasp_validation_summary.csv", index=False)
    display(vasp_summary_df.style.format(precision=4))
    display(vasp_candidate_df.sort_values(["dataset", "val_eta"], ascending=[True, False]).style.format(precision=4))


Seed set to 343
2026-07-28 14:32:51.585 | INFO     | omnixas.model.xasblock_regressor:load:265 - Loading model from /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_plateau_lr5e-4_do010_seed44/best-model-epoch=279-val_median_mse=0.0022.ckpt
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Fine-tuning UniversalXAS on VASP-tuned encoder features: /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/vasp_encoder_plus_universal_head/vasp_encoder_plus_unfrozen_universal_head_seed243/heads/universalXAS/All_VASP_after_encoder/runs/all_vasp_cosT600_lr4e-4_do025_seed343


/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/vasp_encoder_plus_universal_head/vasp_encoder_plus_unfrozen_universal_head_seed243/heads/universalXAS/All_VASP_after_encoder/runs/all_vasp_cosT600_lr4e-4_do025_seed343 exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type     | Params | Mode  | FLOPs
---------------------------------------------------
0 | loss  | MSELoss  | 0      | train | 0    
1 | model | XASBlock | 639 K  | train | 0    
---------------------------------------------------
639 K     Trainable params
0         Non-trainable params
639 K     Total params
2.557     Total estimated model params size (MB)
16        Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |                                                                                | 0/? [00:00…

/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.
/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.


...

Validation: |                                                                                     | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

2026-07-28 14:44:36.721 | INFO     | omnixas.model.xasblock_regressor:fit:246 - Best models saved at /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/vasp_encoder_plus_universal_head/vasp_encoder_plus_unfrozen_universal_head_seed243/heads/universalXAS/All_VASP_after_encoder/runs/all_vasp_plateau_lr3e-4_do015_seed344/best-model-epoch=119-val_median_mse=0.0108.ckpt
2026-07-28 14:44:36.721 | INFO     | omnixas.model.xasblock_regressor:fit:247 - Best validation loss: 0.026672525331377983


,setting,checkpoint,dropout,lr,scheduler,mean_val_eta
0,all_vasp_cosT600_lr4e-4_do025_seed343,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/vasp_encoder_plus_universal_head/vasp_encoder_plus_unfrozen_universal_head_seed243/heads/universalXAS/All_VASP_after_encoder/runs/all_vasp_cosT600_lr4e-4_do025_seed343/best-model-epoch=485-val_median_mse=0.0099.ckpt,0.2500,0.0004,cosine,6.5458
1,all_vasp_plateau_lr3e-4_do015_seed344,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/vasp_encoder_plus_universal_head/vasp_encoder_plus_unfrozen_universal_head_seed243/heads/universalXAS/All_VASP_after_encoder/runs/all_vasp_plateau_lr3e-4_do015_seed344/best-model-epoch=119-val_median_mse=0.0108.ckpt,0.1500,0.0003,plateau,6.0517


,dataset,setting,dropout,lr,scheduler,checkpoint,source_head_checkpoint,source_vasp_e2e_checkpoint,val_median_mse,val_baseline_median_mse,val_eta,test_median_mse,test_baseline_median_mse,test_eta
1,Cu_VASP,all_vasp_cosT600_lr4e-4_do025_seed343,0.2500,0.0004,cosine,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/vasp_encoder_plus_universal_head/vasp_encoder_plus_unfrozen_universal_head_seed243/heads/universalXAS/All_VASP_after_encoder/runs/all_vasp_cosT600_lr4e-4_do025_seed343/best-model-epoch=485-val_median_mse=0.0099.ckpt,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_plateau_lr5e-4_do010_seed44/best-model-epoch=279-val_median_mse=0.0022.ckpt,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/vasp_encoder_plus_universal_head/vasp_encoder_plus_unfrozen_universal_head_seed243/best_vasp_encoder_plus_universal_head.ckpt,0.0027,0.0209,7.6275,0.0028,0.0229,8.1331
3,Cu_VASP,all_vasp_plateau_lr3e-4_do015_seed344,0.1500,0.0003,plateau,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/vasp_encoder_plus_universal_head/vasp_encoder_plus_unfrozen_universal_head_seed243/heads/universalXAS/All_VASP_after_encoder/runs/all_vasp_plateau_lr3e-4_do015_seed344/best-model-epoch=119-val_median_mse=0.0108.ckpt,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_plateau_lr5e-4_do010_seed44/best-model-epoch=279-val_median_mse=0.0022.ckpt,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/vasp_encoder_plus_universal_head/vasp_encoder_plus_unfrozen_universal_head_seed243/best_vasp_encoder_plus_universal_head.ckpt,0.0030,0.0209,6.9360,0.0032,0.0229,7.2636
0,Ti_VASP,all_vasp_cosT600_lr4e-4_do025_seed343,0.2500,0.0004,cosine,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/vasp_encoder_plus_universal_head/vasp_encoder_plus_unfrozen_universal_head_seed243/heads/universalXAS/All_VASP_after_encoder/runs/all_vasp_cosT600_lr4e-4_do025_seed343/best-model-epoch=485-val_median_mse=0.0099.ckpt,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_plateau_lr5e-4_do010_seed44/best-model-epoch=279-val_median_mse=0.0022.ckpt,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/vasp_encoder_plus_universal_head/vasp_encoder_plus_unfrozen_universal_head_seed243/best_vasp_encoder_plus_universal_head.ckpt,0.0290,0.1585,5.4642,0.0268,0.1707,6.3809
2,Ti_VASP,all_vasp_plateau_lr3e-4_do015_seed344,0.1500,0.0003,plateau,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/vasp_encoder_plus_universal_head/vasp_encoder_plus_unfrozen_universal_head_seed243/heads/universalXAS/All_VASP_after_encoder/runs/all_vasp_plateau_lr3e-4_do015_seed344/best-model-epoch=119-val_median_mse=0.0108.ckpt,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/heads/universalXAS/All_FEFF/runs/feff_plateau_lr5e-4_do010_seed44/best-model-epoch=279-val_median_mse=0.0022.ckpt,/mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_new_universal_runs/vasp_encoder_plus_universal_head/vasp_encoder_plus_unfrozen_universal_head_seed243/best_vasp_encoder_plus_universal_head.ckpt,0.0307,0.1585,5.1673,0.0295,0.1707,5.7819


## Notes

- The first UniversalXAS/FEFF stages use only exported `features/*.txt`; they do not need raw structures.
- The VASP encoder stage needs `OMNIXAS_DATA_ROOT` because it rebuilds graphs from raw POSCAR/FEFF inputs.
- The VASP encoder stage deliberately trains both `encoder` and `head`; the notebook asserts the UniversalXAS head is trainable.
- `RUN_FINAL_TEST=True` here because this notebook is configured as a full unattended batch run.
